# 22a. Chuẩn bị demo: cắt 3 đoạn D1, D2, D3 và chạy bộ phát hiện trên từng đoạn (GĐ J, task 8.2), CPU
Cắt 3 đoạn demo (mục 3 của `report/KICH_BAN_DEMO.md`) từ `NF-ToN-IoT-v3.csv` thành file nhỏ (giữ đủ flow, không lấy mẫu); kiểm tra phần trước mốc tấn công
chỉ có lưu lượng bình thường (nếu không thì dời mốc bắt đầu và ghi lại); chạy `scripts/detector.py` trên từng đoạn với hybrid chính và hybrid nhẹ
(có đối chiếu đồ thị offline để biết nhãn thật); so thời gian phát hiện với kịch bản; xuất `demo_replay.json` cho trang `report/demo/demo_replay.html`
và 3 file đoạn cho notebook `22_demo`. Toàn bộ phần tính toán nằm trong `scripts/demo_prep.py`, chạy bằng tiến trình con như các notebook khác.
Không sửa mô hình, cấu hình, ngưỡng đã đóng băng.

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

SEED = 42
WORK = "/kaggle/working"
TMP = "/tmp/prep"
REPO = "/tmp/gnn-iot-ids"
URL = "https://github.com/minucedev/gnn-iot-ids.git"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None, "seed": SEED}
T_START = time.time()


def cpu_model():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return "không rõ"


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1, default=str)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), log=None, **kw):
    """Chạy lệnh, in kết quả theo từng dòng ngay khi có (và ghi vào `log`), để thấy tiến độ và thấy cả phần đã chạy khi lệnh lỗi."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1, **kw)
    lines = []
    logf = open(log, "w", encoding="utf-8") if log else None
    for line in p.stdout:
        print(line, end="", flush=True)
        lines.append(line)
        if logf:
            logf.write(line)
            logf.flush()
    p.wait()
    if logf:
        logf.close()
    print(f"[mã thoát {p.returncode}]")
    if p.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return "".join(lines)


def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", URL, REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])
INFO["cpu"], INFO["cpu_count"] = cpu_model(), os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| CPU:", INFO["cpu"])

## pytest (dừng nếu lỗi): chạy mặc định của detector vẫn y hệt, các tùy chọn mới, đóng gói demo

In [ ]:
with step("pytest"):
    out = run([sys.executable, "-m", "pytest", "-q", "tests/test_detector_stream.py", "tests/test_demo_replay.py", "tests/test_stream_features.py",
               "tests/test_realtime_eval.py", "tests/test_build_graph.py"], cwd=REPO)
    INFO["pytest"] = out.strip().splitlines()[-1]
save_info()

## Đường dẫn đầu vào (như notebook 20)

In [ ]:
DS = "nf_ton_iot_v3"
OLD = f"{REPO}/reports/kaggle"
SPEC = {DS: dict(data=os.path.dirname(find(f"{DS}/split.json")),
                 ckpt=os.path.dirname(find(f"ckpt/{DS}_gat_s42.pt")),
                 folds=os.path.dirname(os.path.dirname(find(f"models_hybrid/{DS}_s42/lgbm_oofavg_il.pkl"))),
                 light=os.path.dirname(os.path.dirname(find(f"models_light/{DS}_s42/lgbm_n100_l15.pkl"))),
                 baseline_search=f"{OLD}/gnn-iot-ids-05-baselines/baseline_search_{DS}.json",
                 ablation_search=f"{OLD}/gnn-iot-ids-16d-ablation-ton/ablation_search_{DS}.json",
                 gnn_csv=f"{OLD}/gnn-iot-ids-07-train-gnn/results_gnn_{DS}.csv",
                 hybrid_csv=f"{OLD}/gnn-iot-ids-15f-hybrid-il-ton/results_hybrid_oofavg_il_{DS}.csv")}
CFG = dict(dataset=DS, split="test", flows_csv=find("NF-ToN-IoT-v3.csv"), expect_commit=INFO["commit"][:7],
           thresholds=f"{OLD}/gnn-iot-ids-16c-aggregate/thresholds.json",
           thresholds_c={DS: f"{OLD}/gnn-iot-ids-16h-c-threshold/thresholds_c_{DS}.json"}, datasets=SPEC)
for k, v in SPEC[DS].items():
    assert os.path.exists(v), f"{k}: không thấy {v}"
for p in (CFG["thresholds"], *CFG["thresholds_c"].values(), CFG["flows_csv"]):
    assert os.path.exists(p), p
CFG_PATH = f"{WORK}/detector_config.json"
json.dump(CFG, open(CFG_PATH, "w"), ensure_ascii=False, indent=1)
INFO["config"] = CFG
save_info()
print("CSV:", CFG["flows_csv"], f"{os.path.getsize(CFG['flows_csv']) / 2**30:.2f} GB")

## Cắt đoạn, kiểm tra, chạy bộ phát hiện, xuất demo_replay.json (`scripts/demo_prep.py`)

In [ ]:
with step("demo_prep"):
    run([sys.executable, "-I", "scripts/demo_prep.py", "--config", CFG_PATH, "--out", WORK, "--tmp", TMP, "--seed", str(SEED), "--commit", INFO["commit"]],
        log=f"{WORK}/demo_prep_log.txt", cwd=REPO)

## Kết quả

In [ ]:
print(open(f"{WORK}/segments_result.csv", encoding="utf-8").read())
prep = json.load(open(f"{WORK}/demo_prep_info.json", encoding="utf-8"))
INFO["result_table"], INFO["replay_json_kb"], INFO["prep_steps_s"] = prep["result_table"], prep["replay_json_kb"], prep["steps_s"]
save_info()
print(sorted(os.listdir(WORK)), sorted(os.listdir(f"{WORK}/demo_data")))